# 01 — BioKG training graph and recorded relationships

Explore the actual **official training split** used by the research demo. Local entity indices are
unique only within their type; `drug:0` and `protein:0` are different entities. Recorded edges,
uncalibrated model scores, and clinical evidence are separate concepts.

No predictive performance or clinical conclusions are claimed here. The default run skips data
access. Enable the official data cells explicitly; there is no synthetic fallback. Saved outputs
have been cleared so historical counts and plots cannot be mistaken for current results.

Sources: [official OGB BioKG protocol](https://ogb.stanford.edu/docs/linkprop/#ogbl-biokg).
OGB uses a random transductive split and typed head/tail ranking. This is not a temporal or unseen-drug evaluation.


## 1. Setup and data access

Use a Python 3.11 kernel from this repository. Install CPU PyTorch first, then `requirements.txt`
and `requirements-notebooks.txt` in the terminal. No notebook cell installs packages or changes
PyTorch loading globally. Initial official data preparation needs several GB of disk space.


In [ ]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
import torch
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/data/load_biokg.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or its notebooks folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data.load_biokg import load_dataset, load_biokg_as_hetero
from src.utils.graph_contract import graph_contract, flatten_graph, query_indices

# Set BIOKG_NOTEBOOK_RUN_DATA=1 only when official data access/disk are available.
# Default execution verifies setup and skips data-dependent cells explicitly.
RUN_DATA = os.getenv("BIOKG_NOTEBOOK_RUN_DATA", "0") == "1"
dataset = graph = contract = None
print({"python": sys.version.split()[0], "torch": torch.__version__,
       "pandas": pd.__version__, "official_data_enabled": RUN_DATA})


In [ ]:
if RUN_DATA:
    dataset = load_dataset()  # Downloads official OGB data only if not cached.
    graph = load_biokg_as_hetero(dataset)  # Uses dataset.load_split("train") only.
    contract = graph_contract(graph)
    print({"train_graph_sha256": contract["train_graph_sha256"],
           "node_types": contract["node_types"], "relations": len(contract["relations"])})
else:
    print("SKIPPED: official BioKG data is disabled; no substitute data is used.")


## 2. Entity identities and relation counts

The shared contract fixes type order, disjoint offsets, relation order and the exact training
edge digest. Do not apply `ToUndirected` or concatenate type-local indices without offsets.
Relation IDs remain recorded identifiers; this notebook does not invent biological labels.


In [ ]:
if graph is not None:
    node_table = pd.DataFrame([
        {"entity_type": kind, "nodes": contract["num_nodes"][kind],
         "global_offset": contract["offsets"][kind]}
        for kind in contract["node_types"]
    ])
    display(node_table.style.bar(subset=["nodes"], color="#c9dded"))
    relation_table = pd.DataFrame([
        {"source_type": src, "relation_id": rel, "target_type": dst,
         "training_edges": int(graph[(src, rel, dst)].edge_index.shape[1])}
        for src, rel, dst in contract["relations"]
    ])
    display(relation_table.sort_values("training_edges", ascending=False)
            .style.bar(subset=["training_edges"], color="#c9dded"))
else:
    print("SKIPPED: entity and relation tables require official BioKG data.")


## 3. Official train, validation and test boundaries

The graph constructor consumes training triples only. The optional full audit checks that the
message-passing graph equals the official training triples and that exact typed held-out triples
are absent. Audit results do not establish independence of reversed pairs or equivalent facts.
Reading split counts/auditing identities here does not perform model selection or scoring.


In [ ]:
if dataset is not None:
    rows = []
    for name in ("train", "valid", "test"):
        split = dataset.load_split(name)
        rows.append({"split": name, "triples": len(split["head"]),
                     "used_for_message_passing": name == "train",
                     "provided_head_negatives": "head_neg" in split,
                     "provided_tail_negatives": "tail_neg" in split})
        del split
    display(pd.DataFrame(rows))
else:
    print("SKIPPED: split counts require official data.")

RUN_SPLIT_AUDIT = os.getenv("BIOKG_NOTEBOOK_AUDIT_SPLITS", "0") == "1"
if RUN_SPLIT_AUDIT and dataset is not None:
    from src.utils.split_audit import audit_split_separation
    split_audit = audit_split_separation(dataset, graph, contract)
    print(json.dumps(split_audit, indent=2))
else:
    print("SKIPPED: full exact-triple audit; set BIOKG_NOTEBOOK_AUDIT_SPLITS=1 with data enabled.")


## 4. Drug-drug records and bounded neighborhoods

The same helper as `/graph_data` returns at most 100 recorded incident edges. These edges are
neither model attributions nor importance scores. The returned neighborhood need not include a
direct link between the two drugs. Optional DrugBank lookups enrich metadata; they do not alter
the training graph, scores, or evidence for clinical harm.


In [ ]:
if graph is not None:
    from src.utils.mapping_loader import load_entity_mappings
    from src.utils.paths import BIOKG_MAPPING_DIR
    from src.utils.neighborhood import source_neighborhood
    ddi = relation_table.query("source_type == 'drug' and target_type == 'drug'")
    display(ddi)
    mappings = load_entity_mappings(BIOKG_MAPPING_DIR)
    neighborhood = source_neighborhood(graph, mappings, 58, 89, limit=100)
    print({"view_type": neighborhood["view_type"], "truncated": neighborhood["truncated"],
           "edge_limit": neighborhood["edge_limit"], "description": neighborhood["description"]})
    display(pd.DataFrame(neighborhood["nodes"]))
    display(pd.DataFrame(neighborhood["links"]))
else:
    print("SKIPPED: recorded neighborhoods require the official training graph.")


## 5. Optional export of training records

Export uses type-local drug indices and includes split provenance. It excludes validation and
test records. Publishing an export requires reviewing the source dataset terms and documenting
its scope. This cell writes only when `EXPORT_TRAIN_DDI` is enabled explicitly.


In [ ]:
EXPORT_TRAIN_DDI = False
if EXPORT_TRAIN_DDI:
    if graph is None:
        raise RuntimeError("Enable official BioKG data before exporting")
    import csv
    target = ROOT / "artifacts/notebooks/ddi_train_triples.csv"
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open("w", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(["head_type", "head_local_index", "relation_id", "tail_type",
                         "tail_local_index", "split"])
        for src, rel, dst in contract["relations"]:
            if src == dst == "drug":
                for head, tail in graph[(src, rel, dst)].edge_index.T.tolist():
                    writer.writerow([src, head, rel, dst, tail, "train"])
    print(target)
else:
    print("SKIPPED: training-record export is disabled.")


## 6. Continue to model verification

[Notebook 02](02_ogbl_biokg_eda.ipynb) uses the same graph contract, relation-aware DistMult
scoring and checkpoint gate as the application. It separates validation-based training choices
from a final evaluation of fixed weights on test data. Validated explanation methods remain
unimplemented; legacy explanation endpoints stay unavailable.
